# Notebook 02 — Model Training: EfficientNet-B4 + CBAM

**Diabetic Retinopathy Detection** — APTOS 2019 Blindness Detection

This notebook implements the full training pipeline for classifying retinal fundus images into five DR severity grades using a **3-phase progressive fine-tuning** strategy.

| Phase | Strategy | Epochs | LR | Backbone |
|-------|----------|--------|----|----------|
| 1 | Feature extraction | 10 | 1e-3 | Frozen |
| 2 | Partial fine-tuning | 15 | 1e-4 | Top 50% unfrozen |
| 3 | Full fine-tuning | 5 | 1e-5 | Fully unfrozen |

**Key components:**
- **Backbone:** EfficientNet-B4 (pretrained on ImageNet)
- **Attention:** Convolutional Block Attention Module (CBAM)
- **Loss:** 70% WeightedFocalLoss + 30% LabelSmoothingCE
- **Optimizer:** AdamW with layer-wise learning rates
- **Scheduler:** Cosine Annealing with Warm Restarts
- **Mixed precision (AMP)** for faster training on GPU

---
## 1. Environment Setup

In [ ]:
import sys
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
%matplotlib inline

# Add project src/ to the Python path
PROJECT_ROOT = Path("../").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import (
    DATASET_CONFIG,
    PREPROCESSING_CONFIG,
    MODEL_CONFIG,
    TRAINING_CONFIG,
    TRAINING_PHASES,
    set_seed,
    get_device,
)
from preprocessing import DRPreprocessor
from dataset import DRDataset, create_data_loaders, get_train_transforms, get_valid_transforms
from models.efficientnet_model import EfficientNetDR, build_model
from training.trainer import DRTrainer
from training.losses import WeightedFocalLoss, LabelSmoothingCrossEntropyLoss

# Reproducibility
SEED = TRAINING_CONFIG["seed"]
set_seed(SEED)

# Device
DEVICE = get_device()
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print(f"Device selected : {DEVICE}")

In [ ]:
# Paths
DATA_DIR   = PROJECT_ROOT / "data" / "aptos2019"
CSV_PATH   = DATA_DIR / "train.csv"
IMAGE_DIR  = DATA_DIR / "train_images"
OUTPUT_DIR = PROJECT_ROOT / "results" / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Image settings
IMG_SIZE = PREPROCESSING_CONFIG["image_size"]  # 512
NUM_CLASSES = DATASET_CONFIG["num_classes"]     # 5
CLASS_NAMES = DATASET_CONFIG["class_names"]     # ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

print(f"CSV path    : {CSV_PATH}")
print(f"Image dir   : {IMAGE_DIR}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"Image size  : {IMG_SIZE}x{IMG_SIZE}")
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

---
## 2. Dataset Loading & Stratified Split

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Total samples: {len(df)}")
print(f"Columns      : {list(df.columns)}")
df.head(10)

In [ ]:
# Stratified 80/20 train-validation split
train_df, valid_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["diagnosis"],
)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print(f"Training   samples: {len(train_df)}")
print(f"Validation samples: {len(valid_df)}")
print(f"Split ratio       : {len(train_df)/len(df)*100:.1f}% / {len(valid_df)/len(df)*100:.1f}%")

In [ ]:
# Class distribution comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

colors = ["#2196F3", "#4CAF50", "#FF9800", "#F44336", "#9C27B0"]

# Full dataset
counts_all = df["diagnosis"].value_counts().sort_index()
axes[0].bar(CLASS_NAMES, counts_all.values, color=colors, edgecolor="white", linewidth=0.8)
axes[0].set_title("Full Dataset", fontweight="bold")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts_all.values):
    axes[0].text(i, v + 10, str(v), ha="center", fontweight="bold", fontsize=9)

# Train split
counts_train = train_df["diagnosis"].value_counts().sort_index()
axes[1].bar(CLASS_NAMES, counts_train.values, color=colors, edgecolor="white", linewidth=0.8)
axes[1].set_title(f"Training Set ({len(train_df)})", fontweight="bold")
axes[1].set_ylabel("Count")
for i, v in enumerate(counts_train.values):
    axes[1].text(i, v + 10, str(v), ha="center", fontweight="bold", fontsize=9)

# Validation split
counts_val = valid_df["diagnosis"].value_counts().sort_index()
axes[2].bar(CLASS_NAMES, counts_val.values, color=colors, edgecolor="white", linewidth=0.8)
axes[2].set_title(f"Validation Set ({len(valid_df)})", fontweight="bold")
axes[2].set_ylabel("Count")
for i, v in enumerate(counts_val.values):
    axes[2].text(i, v + 10, str(v), ha="center", fontweight="bold", fontsize=9)

for ax in axes:
    ax.set_xlabel("DR Grade")
    ax.tick_params(axis="x", rotation=25)

plt.suptitle("Class Distribution — Stratified 80/20 Split", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Percentage table
dist_table = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Train Count": counts_train.values,
    "Train %": (counts_train.values / len(train_df) * 100).round(1),
    "Val Count": counts_val.values,
    "Val %": (counts_val.values / len(valid_df) * 100).round(1),
})
dist_table

---
## 3. Data Augmentation Preview

Visualise the training augmentation pipeline applied to a single retinal image. The augmentations include random rotation, flips, shift-scale-rotate, noise, brightness/contrast jitter, and coarse dropout.

In [ ]:
# Pick one sample image
sample_row = train_df.iloc[0]
sample_id = sample_row["id_code"]
sample_label = sample_row["diagnosis"]

# Try common extensions
sample_path = None
for ext in [".png", ".jpg", ".jpeg"]:
    candidate = IMAGE_DIR / f"{sample_id}{ext}"
    if candidate.exists():
        sample_path = candidate
        break

if sample_path is None:
    print(f"WARNING: Could not find image for {sample_id}, trying first available image...")
    available = list(IMAGE_DIR.glob("*.*"))
    if available:
        sample_path = available[0]
    else:
        raise FileNotFoundError(f"No images found in {IMAGE_DIR}")

# Load and preprocess
preprocessor = DRPreprocessor(img_size=IMG_SIZE)
sample_img = preprocessor.preprocess(str(sample_path), normalize=False).astype(np.uint8)

print(f"Sample  : {sample_id}")
print(f"Label   : {sample_label} ({CLASS_NAMES[sample_label]})")
print(f"Shape   : {sample_img.shape}")

In [ ]:
# Generate multiple augmented views
train_transform = get_train_transforms(IMG_SIZE)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Original (top-left)
axes[0, 0].imshow(sample_img)
axes[0, 0].set_title("Original", fontweight="bold", fontsize=12)
axes[0, 0].axis("off")

# 7 augmented versions
for idx, ax in enumerate(axes.flat[1:]):
    augmented = train_transform(image=sample_img)
    img_tensor = augmented["image"]  # (C, H, W)
    # Denormalize for display
    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = (img_np * IMAGENET_STD + IMAGENET_MEAN).clip(0, 1)
    ax.imshow(img_np)
    ax.set_title(f"Augmented #{idx + 1}", fontsize=11)
    ax.axis("off")

plt.suptitle(
    f"Training Augmentation Preview \u2014 {sample_id} (Class {sample_label}: {CLASS_NAMES[sample_label]})",
    fontsize=14, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.show()

---
## 4. Model Architecture

Build the **EfficientNet-B4 + CBAM** model and inspect its architecture.

In [ ]:
model = build_model(
    architecture=MODEL_CONFIG["architecture"],   # efficientnet_b4
    num_classes=NUM_CLASSES,                      # 5
    pretrained=MODEL_CONFIG["pretrained"],        # True
    dropout=MODEL_CONFIG["dropout"],              # 0.4
    use_attention=True,
    freeze_backbone=False,
    device=DEVICE,
)

In [ ]:
# Detailed parameter breakdown
param_info = model.count_parameters()

backbone_total    = sum(p.numel() for p in model.backbone.parameters())
attention_total   = sum(p.numel() for p in model.attention.parameters()) if model.use_attention else 0
classifier_total  = sum(p.numel() for p in model.classifier.parameters())

print("=" * 55)
print("  Parameter Breakdown")
print("=" * 55)
print(f"  Backbone (EfficientNet-B4) : {backbone_total:>12,}")
print(f"  CBAM Attention Module      : {attention_total:>12,}")
print(f"  Classification Head        : {classifier_total:>12,}")
print(f"  {'':->53}")
print(f"  Total Parameters           : {param_info['total']:>12,}")
print(f"  Trainable Parameters       : {param_info['trainable']:>12,}")
print(f"  Frozen Parameters          : {param_info['frozen']:>12,}")
print("=" * 55)

# Visualize proportions
fig, ax = plt.subplots(figsize=(8, 4))
components = ["Backbone", "CBAM", "Classifier"]
values = [backbone_total, attention_total, classifier_total]
bars = ax.barh(components, values, color=["#1976D2", "#E64A19", "#388E3C"], edgecolor="white")
for bar, val in zip(bars, values):
    ax.text(bar.get_width() + max(values) * 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:,} ({val / param_info['total'] * 100:.1f}%)",
            va="center", fontsize=10)
ax.set_xlabel("Number of Parameters")
ax.set_title("Model Parameter Distribution", fontweight="bold")
plt.tight_layout()
plt.show()

---
## 5. Training Configuration

Display the 3-phase progressive fine-tuning configuration.

In [ ]:
trainer_config = {
    "num_classes": NUM_CLASSES,
    "num_epochs": 30,
    "batch_size": TRAINING_CONFIG["batch_size"],
    "use_amp": TRAINING_CONFIG["use_amp"],
    "gradient_clip": 1.0,
    "weight_decay": TRAINING_CONFIG["weight_decay"],
    "focal_gamma": TRAINING_CONFIG["focal_gamma"],
    "early_stopping_patience": 7,
    "early_stopping_min_delta": 1e-4,
    "phases": {
        "phase1": {
            "name": "Feature Extraction",
            "epochs": 10,
            "lr": 1e-3,
            "freeze_backbone": True,
            "unfreeze_fraction": 0.0,
            "scheduler_T0": 10,
            "scheduler_Tmult": 1,
        },
        "phase2": {
            "name": "Partial Fine-tuning",
            "epochs": 15,
            "lr": 1e-4,
            "freeze_backbone": False,
            "unfreeze_fraction": 0.5,
            "scheduler_T0": 10,
            "scheduler_Tmult": 2,
        },
        "phase3": {
            "name": "Full Fine-tuning",
            "epochs": 5,
            "lr": 1e-5,
            "freeze_backbone": False,
            "unfreeze_fraction": 1.0,
            "scheduler_T0": 5,
            "scheduler_Tmult": 1,
        },
    },
}

# Pretty-print phase table
print("=" * 75)
print("  3-Phase Progressive Fine-Tuning Schedule")
print("=" * 75)
print(f"  {'Phase':<8} {'Strategy':<24} {'Epochs':>6}  {'LR':>10}  {'Backbone':<20}")
print(f"  {'-'*70}")
for key, phase in trainer_config["phases"].items():
    phase_num = key[-1]
    if phase["freeze_backbone"]:
        backbone_str = "Frozen"
    elif phase["unfreeze_fraction"] < 1.0:
        backbone_str = f"Top {phase['unfreeze_fraction']*100:.0f}% unfrozen"
    else:
        backbone_str = "Fully unfrozen"
    print(f"  {phase_num:<8} {phase['name']:<24} {phase['epochs']:>6}  {phase['lr']:>10.0e}  {backbone_str:<20}")
print("=" * 75)

total_epochs = sum(p["epochs"] for p in trainer_config["phases"].values())
print(f"\n  Total epochs    : {total_epochs}")
print(f"  Batch size      : {trainer_config['batch_size']}")
print(f"  Mixed precision : {trainer_config['use_amp']}")
print(f"  Gradient clip   : {trainer_config['gradient_clip']}")
print(f"  Weight decay    : {trainer_config['weight_decay']}")
print(f"  Focal gamma     : {trainer_config['focal_gamma']}")
print(f"  Early stopping  : patience={trainer_config['early_stopping_patience']}")

---
## 6. Training Execution

Create data loaders and run the full 3-phase progressive fine-tuning.

In [ ]:
# Create data loaders
BATCH_SIZE  = trainer_config["batch_size"]
NUM_WORKERS = TRAINING_CONFIG.get("num_workers", 4)

train_loader, valid_loader = create_data_loaders(
    train_df=train_df,
    valid_df=valid_df,
    train_dir=str(IMAGE_DIR),
    valid_dir=str(IMAGE_DIR),
    batch_size=BATCH_SIZE,
    img_size=IMG_SIZE,
    num_workers=NUM_WORKERS,
)

print(f"Train loader : {len(train_loader)} batches  ({len(train_loader.dataset)} samples)")
print(f"Valid loader : {len(valid_loader)} batches  ({len(valid_loader.dataset)} samples)")
print(f"Batch size   : {BATCH_SIZE}")
print(f"Num workers  : {NUM_WORKERS}")

# Quick sanity check: load one batch
images, labels = next(iter(train_loader))
print(f"\nSanity check:")
print(f"  Batch images shape : {images.shape}")
print(f"  Batch labels shape : {labels.shape}")
print(f"  Labels             : {labels.tolist()[:8]}...")
print(f"  Pixel range        : [{images.min():.3f}, {images.max():.3f}]")

In [ ]:
# Build a fresh model for training (backbone frozen initially by DRTrainer)
model = build_model(
    architecture=MODEL_CONFIG["architecture"],
    num_classes=NUM_CLASSES,
    pretrained=True,
    dropout=MODEL_CONFIG["dropout"],
    use_attention=True,
    freeze_backbone=False,
    device=DEVICE,
)

# Initialize trainer
trainer = DRTrainer(
    model=model,
    config=trainer_config,
    device=DEVICE,
    output_dir=str(OUTPUT_DIR),
)

print("Trainer ready. Starting training...")

In [ ]:
# Run full 3-phase training
history = trainer.train(
    train_loader=train_loader,
    valid_loader=valid_loader,
)

---
## 7. Training Curves

Plot loss, accuracy, and Quadratic Weighted Kappa (QWK) across all training epochs.

In [ ]:
# Load history from file (works even if training was run in a separate session)
history_path = OUTPUT_DIR / "training_history.json"

if history_path.exists():
    with open(history_path) as f:
        history = json.load(f)
    print(f"Loaded training history: {len(history['train_loss'])} epochs")
else:
    print("Using in-memory training history from current session.")

epochs_range = range(1, len(history["train_loss"]) + 1)

# Phase boundary markers
p1_end = trainer_config["phases"]["phase1"]["epochs"]
p2_end = p1_end + trainer_config["phases"]["phase2"]["epochs"]

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Helper to draw phase boundaries
def add_phase_lines(ax):
    for boundary, label in [(p1_end + 0.5, "P1\u2192P2"), (p2_end + 0.5, "P2\u2192P3")]:
        if boundary < len(history["train_loss"]):
            ax.axvline(x=boundary, color="gray", linestyle="--", alpha=0.6, linewidth=1)
            ax.text(boundary, ax.get_ylim()[1] * 0.97, label,
                    ha="center", va="top", fontsize=8, color="gray",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.8))

# 1) Loss
ax = axes[0, 0]
ax.plot(epochs_range, history["train_loss"], "o-", label="Train Loss", markersize=3, linewidth=1.5)
ax.plot(epochs_range, history["val_loss"],   "s-", label="Val Loss",   markersize=3, linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Loss vs Epoch", fontweight="bold")
ax.legend()
add_phase_lines(ax)

# 2) Accuracy
ax = axes[0, 1]
ax.plot(epochs_range, history["train_accuracy"], "o-", label="Train Accuracy", markersize=3, linewidth=1.5)
ax.plot(epochs_range, history["val_accuracy"],   "s-", label="Val Accuracy",   markersize=3, linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy vs Epoch", fontweight="bold")
ax.legend()
add_phase_lines(ax)

# 3) QWK
ax = axes[1, 0]
ax.plot(epochs_range, history["train_qwk"], "o-", label="Train QWK", markersize=3, linewidth=1.5, color="#E64A19")
ax.plot(epochs_range, history["val_qwk"],   "s-", label="Val QWK",   markersize=3, linewidth=1.5, color="#1565C0")
ax.set_xlabel("Epoch")
ax.set_ylabel("Quadratic Weighted Kappa")
ax.set_title("QWK vs Epoch", fontweight="bold")
ax.legend()
add_phase_lines(ax)

# 4) Learning Rate
ax = axes[1, 1]
ax.plot(epochs_range, history["learning_rate"], "o-", markersize=3, linewidth=1.5, color="#388E3C")
ax.set_xlabel("Epoch")
ax.set_ylabel("Learning Rate")
ax.set_title("Learning Rate Schedule", fontweight="bold")
ax.set_yscale("log")
add_phase_lines(ax)

plt.suptitle("Training Curves \u2014 EfficientNet-B4 + CBAM (3-Phase)", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 8. Results Summary

Load the best checkpoint and display final training metrics.

In [ ]:
# Load best checkpoints
results = {}

for tag in ["best_qwk", "best_acc", "last"]:
    ckpt_path = OUTPUT_DIR / f"model_{tag}.pth"
    if ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location="cpu")
        results[tag] = {
            "epoch": ckpt.get("epoch", "?"),
            "val_loss": ckpt.get("metrics", {}).get("loss", ckpt.get("best_val_loss", "N/A")),
            "val_accuracy": ckpt.get("metrics", {}).get("accuracy", ckpt.get("best_val_acc", "N/A")),
            "val_qwk": ckpt.get("metrics", {}).get("qwk", ckpt.get("best_qwk", "N/A")),
            "val_auc_roc": ckpt.get("metrics", {}).get("auc_roc", "N/A"),
        }
        print(f"Loaded {tag:>10} checkpoint (epoch {results[tag]['epoch']})")
    else:
        print(f"Checkpoint not found: {ckpt_path}")

In [ ]:
# Build results table
if results:
    rows = []
    for tag, metrics in results.items():
        row = {
            "Checkpoint": tag,
            "Epoch": metrics["epoch"],
        }
        for k in ["val_loss", "val_accuracy", "val_qwk", "val_auc_roc"]:
            v = metrics[k]
            row[k] = f"{v:.4f}" if isinstance(v, (int, float)) else str(v)
        rows.append(row)

    results_df = pd.DataFrame(rows)
    results_df.columns = ["Checkpoint", "Epoch", "Val Loss", "Val Accuracy", "Val QWK", "Val AUC-ROC"]

    print("=" * 80)
    print("  Final Results Summary")
    print("=" * 80)
    display(results_df.style.set_properties(**{"text-align": "center"}).hide(axis="index"))
else:
    print("No checkpoints found. Run the training cells first.")

In [ ]:
# Per-class accuracy from the best QWK checkpoint
best_ckpt_path = OUTPUT_DIR / "model_best_qwk.pth"
if best_ckpt_path.exists():
    ckpt = torch.load(best_ckpt_path, map_location="cpu")
    best_metrics = ckpt.get("metrics", {})

    per_class = []
    for i in range(NUM_CLASSES):
        acc = best_metrics.get(f"class_{i}_acc", None)
        per_class.append(acc)

    if per_class[0] is not None:
        fig, ax = plt.subplots(figsize=(10, 5))
        bars = ax.bar(CLASS_NAMES, per_class, color=colors, edgecolor="white", linewidth=0.8)
        for bar, val in zip(bars, per_class):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f"{val:.1%}", ha="center", fontweight="bold", fontsize=11)
        ax.set_ylim(0, 1.15)
        ax.set_ylabel("Accuracy")
        ax.set_xlabel("DR Grade")
        ax.set_title("Per-Class Accuracy (Best QWK Checkpoint)", fontweight="bold")
        ax.axhline(y=np.mean(per_class), color="red", linestyle="--", alpha=0.6, label=f"Mean: {np.mean(per_class):.1%}")
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("Per-class accuracy not available in checkpoint metrics.")
else:
    print("Best QWK checkpoint not found. Run training first.")

In [ ]:
# Training time summary
if "epoch_time" in history and history["epoch_time"]:
    total_time_min = sum(history["epoch_time"]) / 60
    avg_epoch_sec  = np.mean(history["epoch_time"])
    print(f"Total training time  : {total_time_min:.1f} minutes")
    print(f"Avg epoch time       : {avg_epoch_sec:.1f} seconds")

print("\n" + "=" * 60)
print("  Training notebook complete.")
print("  Checkpoints saved to:", OUTPUT_DIR)
print("=" * 60)